# L3 — I3-4 지연 측정: F1 base + I2 함께 돌린 판정 1회

## 왜 하나

I3(F1 base + I2 점수 평균)는 요약·전체 탐지율 기준을 모두 넘어 **조건부 채택**됐다. 남은 조건은 **I3-4: 두 판정기를 함께 돌린 판정 1회 p95 ≤ 68ms(A100)**.
판정기 하나가 각각 약 40ms(p95, A100)라 차례로 돌리면 예산(8토큰 × 8.5ms = 68ms)을 넘을 가능성이 크다. GPU에서 두 판정기를 **동시에** 돌리면 들어오는지 잰다.

## 무엇을 하나

- 판정기 코드는 게이트웨이의 `hallugate/judge.py` 그대로, 가중치는 드라이브 `final/judge_base`(F1 base)와 `i2/judge_i2`(I2)
- 입력: RAGTruth train 응답 300개의 문장 끝 + 8토큰 지점 중 300개 (F1 base는 문장 앞부분 + 근거 검색, I2는 근거 전체 + 응답 처음부터 그 지점까지)
- 판정 1회 = 두 점수 + 조합. 근거 준비(요청당 1회)는 제외
- 방식: **차례로**(게이트웨이 `judge()` 두 번) / **동시**(스레드 2개, 판정기마다 CUDA 스트림)

## 사전 정의 기준 (결과 보기 전, SSOT §6 L3에 고정)

- 두 방식 중 p95가 작은 방식으로 판정. **I3-4 충족 = A100에서 p95 ≤ 68ms**
- 동시 방식의 점수는 차례로 방식과 같아야 한다(최대 차이 ≤ 0.05)
- A100이 아니면 판정 보류(보고만). 판정기 단독 p50·p95도 보고

> **런타임: A100**. 약 10분. 결과는 드라이브 `Colab Notebooks/HalluGate/l3/`와 `l3_results.zip`.


## 1. 환경 준비

In [ ]:
JUDGE_SRC = '"""문장 판정기.\n\n입력 방식은 판정기 폴더의 `judge_config.json`이 정한다 (SPEC §4).\n  retrieval (F1 base): 근거 → 40단어 조각 → 임베딩 (요청당 1회, prepare)\n                       판정 텍스트 → 임베딩 → 유사한 조각 top-k (원문 순서) → 교차 인코더 [근거 조각, 문장 앞부분]\n  full      (I2식)    : 교차 인코더 [근거 전체, 응답 처음부터 판정 지점까지]. 근거 검색 없음\n점수는 logit(클수록 환각). 임계값은 재판정 간격(stride)별로 설정 파일의 값을 쓴다.\n"""\nimport json\nimport os\nimport threading\nimport time\nfrom dataclasses import dataclass\n\nimport torch\n\n\n@dataclass\nclass Verdict:\n    score: float          # logit, 클수록 환각\n    hallucinated: bool\n    judge_ms: float       # 검색 + 판정 시간\n\n\n@dataclass\nclass RefIndex:\n    chunks: list\n    emb: object           # torch.Tensor [n_chunks, d] (retrieval) / None (full)\n    prepare_ms: float\n    text: str = \'\'        # 근거 원문 (full 방식)\n\n\ndef chunk_text(text, n):\n    w = text.split()\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w), 1), n)] or [text]\n\n\ndef _stride_key(stride):\n    return \'end\' if not stride else str(stride)\n\n\ndef read_config(model_dir):\n    p = os.path.join(model_dir, \'judge_config.json\')\n    return json.load(open(p)) if os.path.isfile(p) else {}\n\n\ndef input_mode(cfg):\n    """설정 파일에서 입력 방식을 정한다. 명시(`input_mode`)가 없으면 I2 노트북 형식(retrieval: null)을 full로 본다."""\n    if cfg.get(\'input_mode\'):\n        return cfg[\'input_mode\']\n    inp = cfg.get(\'input\') or {}\n    return \'full\' if isinstance(inp, dict) and \'retrieval\' in inp and inp[\'retrieval\'] is None else \'retrieval\'\n\n\nclass StudentJudge:\n    def __init__(self, model_dir, embedder=\'sentence-transformers/all-MiniLM-L6-v2\', k=8, max_len=512,\n                 chunk_words=40, threshold=0.0, device=\'\', mode=\'retrieval\', thresholds=None, threshold_key=\'svc10_orig\',\n                 default_stride=0):\n        from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.mode = mode\n        self.tok = AutoTokenizer.from_pretrained(model_dir)\n        cfg = AutoConfig.from_pretrained(model_dir)\n        kw = {}\n        if getattr(cfg, \'model_type\', \'\') == \'modernbert\':\n            # 새 transformers는 reference_compile을 모델 인자로 받지 않으므로 설정에 넣는다\n            if hasattr(cfg, \'reference_compile\'):\n                cfg.reference_compile = False\n            kw[\'attn_implementation\'] = \'sdpa\'\n        # 가중치가 fp16으로 저장돼 있어도 fp32로 올린다 (CPU 판정, 노트북 점수와 일치). GPU에서는 autocast가 반정밀도로 돌린다.\n        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir, config=cfg, **kw).float().to(self.device).eval()\n        if mode == \'retrieval\' and isinstance(embedder, str):\n            from sentence_transformers import SentenceTransformer\n            embedder = SentenceTransformer(embedder, device=self.device)\n        self.embedder = embedder\n        self.k, self.max_len, self.chunk_words, self.threshold = k, max_len, chunk_words, threshold\n        self.thresholds, self.threshold_key = thresholds or {}, threshold_key\n        self.default_stride = default_stride\n        self._lock = threading.Lock()     # 모델 1개를 여러 요청이 공유하므로 직렬화\n\n    def threshold_for(self, stride, key=None):\n        """재판정 간격별 임계값 (설정 파일 thresholds[간격][key]). 없으면 기본 임계값."""\n        return (self.thresholds.get(_stride_key(stride)) or {}).get(key or self.threshold_key, self.threshold)\n\n    def _embed(self, texts):\n        return self.embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True,\n                                    show_progress_bar=False)\n\n    def prepare(self, reference):\n        t0 = time.perf_counter()\n        if self.mode == \'full\':\n            return RefIndex([], None, (time.perf_counter() - t0) * 1000, reference)\n        chunks = chunk_text(reference, self.chunk_words)\n        with self._lock:\n            emb = self._embed(chunks)\n        return RefIndex(chunks, emb, (time.perf_counter() - t0) * 1000, reference)\n\n    def _pair(self, ref, text, prior):\n        if self.mode == \'full\':\n            return ref.text, (prior or \'\') + text\n        e = self._embed([text])[0]\n        top = torch.topk(ref.emb @ e, min(self.k, ref.emb.shape[0])).indices.tolist()\n        return \' ... \'.join(ref.chunks[i] for i in sorted(top)), text\n\n    @torch.no_grad()\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        """text: 판정할 문장(또는 문장 앞부분, 앞뒤 공백 제거), prior: 응답에서 그 문장 앞까지의 원문."""\n        t0 = time.perf_counter()\n        with self._lock:\n            a, b = self._pair(ref, text, prior)\n            enc = self.tok([a], [b], truncation=\'only_first\', max_length=self.max_len,\n                           return_tensors=\'pt\').to(self.device)\n            with torch.autocast(self.device, enabled=(self.device == \'cuda\')):\n                score = float(self.model(**enc).logits.squeeze().float().cpu())\n            if self.device == \'cuda\':\n                torch.cuda.synchronize()\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n\n\ndef load_judge(settings):\n    """판정기 폴더의 judge_config.json을 읽어 StudentJudge를 만든다. 환경 변수로 지정한 값이 설정 파일보다 우선한다."""\n    cfg = read_config(settings.judge_model_dir)\n    pick = lambda env, key, default: env if env is not None else cfg.get(key, default)\n    dflt = (cfg.get(\'default\') or {})\n    kind = dflt.get(\'kind\', \'end\')\n    return StudentJudge(settings.judge_model_dir,\n                        embedder=pick(settings.judge_embedder, \'embedder\', \'sentence-transformers/all-MiniLM-L6-v2\'),\n                        k=pick(settings.judge_k, \'k\', 8), max_len=pick(settings.judge_max_len, \'max_len\', 512),\n                        chunk_words=pick(settings.judge_chunk_words, \'chunk_words\', 40),\n                        threshold=pick(settings.judge_threshold, \'threshold\', 0.0), device=settings.judge_device,\n                        mode=input_mode(cfg),\n                        # 임계값을 환경 변수로 고정하면 간격별 임계값은 쓰지 않는다\n                        thresholds={} if settings.judge_threshold is not None else cfg.get(\'thresholds\', {}),\n                        threshold_key=dflt.get(\'threshold\', \'svc10_orig\'),\n                        default_stride=0 if kind == \'end\' else int(kind))\n\n\nclass LexicalJudge:\n    """모델 없이 파이프라인을 시험하기 위한 대체 판정기 (테스트·데모 전용).\n    근거에 없는 단어 비율을 점수로 쓴다. 판별력은 없다고 봐야 한다."""\n\n    default_stride = 0\n\n    def __init__(self, threshold=0.5, delay_ms=0.0, fail=None):\n        self.threshold, self.delay_ms = threshold, delay_ms\n        self.fail = fail          # 테스트용: 이 함수가 True를 돌려주는 판정에서 예외를 낸다\n\n    def threshold_for(self, stride, key=None):\n        return self.threshold\n\n    def prepare(self, reference):\n        return RefIndex([reference], {w.lower().strip(\'.,\') for w in reference.split()}, 0.0, reference)\n\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        t0 = time.perf_counter()\n        if self.fail is not None and self.fail(text):\n            raise RuntimeError(\'judge failure (test)\')\n        if self.delay_ms:\n            time.sleep(self.delay_ms / 1000)\n        words = [w.lower().strip(\'.,\') for w in text.split() if w.strip(\'.,\')]\n        score = sum(w not in ref.emb for w in words) / max(len(words), 1)\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n'
import os, sys, json, time, random, subprocess, shutil
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'sentence-transformers', 'tiktoken', 'pysbd', '-U', 'transformers>=4.48'], check=True)
subprocess.run('git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || true', shell=True)
import numpy as np, torch
from google.colab import drive
drive.mount('/content/drive')
open('hallugate_judge.py', 'w').write(JUDGE_SRC); sys.path.insert(0, '.')
from hallugate_judge import StudentJudge, read_config, input_mode
OUT = '/content/drive/MyDrive/Colab Notebooks/HalluGate'
F_DIR, I_DIR, L3_DIR = os.path.join(OUT, 'final', 'judge_base'), os.path.join(OUT, 'i2', 'judge_i2'), os.path.join(OUT, 'l3')
os.makedirs(L3_DIR, exist_ok=True)
for d in [F_DIR, I_DIR]:
    assert os.path.exists(os.path.join(d, 'judge_config.json')), f'{d}/judge_config.json 없음'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
COMBO = {'F_mean': -3.6794357299804688, 'F_std': 2.1870224475860596, 'I_mean': -4.456174373626709, 'I_std': 2.400838851928711, 'threshold': 2.991638706756592}
print('device', GPU, '| 조합 상수(I3)', COMBO)

## 2. 판정 입력 300개

In [ ]:
# 판정 입력: RAGTruth train 응답의 문장 끝 + 8토큰 지점 (게이트웨이와 같은 cl100k 토큰, 문장 시작부터)
import pysbd, tiktoken
ENC = tiktoken.get_encoding('cl100k_base')
SEG = pysbd.Segmenter(language='en', clean=False, char_span=True)

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

def points_of(text, s=8):
    # (문장 시작 글자, 지점 글자 끝) 목록: 문장마다 s토큰째 + 문장 끝
    ids = ENC.encode(text, disallowed_special=())
    _, starts = ENC.decode_with_offsets(ids)
    ends = starts[1:] + [len(text)]
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        tok = [e for st, e in zip(starts, ends) if a <= st < sp.end or (st < a < e)]
        for k, e in enumerate(tok, 1):
            if k % s == 0 or k == len(tok):
                out.append((a, min(e, sp.end)))
    return out

def make_inputs(responses, sources, n_resp=300, n_pts=300, seed=2027):
    rs = random.Random(seed)
    pool = [r for r in responses if r['split'] == 'train']
    cands = []
    for r in rs.sample(pool, min(n_resp, len(pool))):
        ref = build_reference(sources[r['source_id']])
        for a, cut in points_of(r['response']):
            text = r['response'][a:cut].strip()
            if text: cands.append({'ref': ref, 'text': text, 'prior': r['response'][:a], 'task': sources[r['source_id']]['task_type']})
    return rs.sample(cands, min(n_pts, len(cands)))

BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses: r['task_type'] = sources[r['source_id']]['task_type']
INPUTS = make_inputs(responses, sources)
print(f"입력 {len(INPUTS)}개 | 과제", {t: sum(x['task'] == t for x in INPUTS) for t in ['QA', 'Summary', 'Data2txt']},
      f"| 응답 앞부분 길이(글자) 중앙값 {np.median([len(x['prior']) + len(x['text']) for x in INPUTS]):.0f}")

## 3. 측정

In [ ]:
import contextlib, threading
from concurrent.futures import ThreadPoolExecutor

def make_judge(d, device):
    # load_judge와 같은 값으로 (판정기 폴더의 judge_config.json)
    cfg = read_config(d)
    return StudentJudge(d, embedder=cfg.get('embedder', 'sentence-transformers/all-MiniLM-L6-v2'), k=cfg.get('k', 8),
                        max_len=cfg.get('max_len', 512), chunk_words=cfg.get('chunk_words', 40), device=device,
                        mode=input_mode(cfg))

def combine(sf, si):
    # I3 조합: 두 점수를 cal-L 멀쩡한 응답 기준으로 표준화해 평균
    return ((sf - COMBO['F_mean']) / COMBO['F_std'] + (si - COMBO['I_mean']) / COMBO['I_std']) / 2

def judge_seq(jF, jI, refF, refI, x):
    # 차례로: 게이트웨이 judge() 두 번 (각각 토크나이즈·검색·순전파·동기화 포함)
    return jF.judge(refF, x['text'], x['prior']).score, jI.judge(refI, x['text'], x['prior']).score

STREAMS = {}
def _score_on_stream(j, ref, text, prior):
    # judge()와 같은 계산을 판정기 전용 CUDA 스트림에서. 동기화는 그 스트림만 (.cpu()가 현재 스트림을 기다림)
    ctx = torch.cuda.stream(STREAMS.setdefault(id(j), torch.cuda.Stream())) if j.device == 'cuda' else contextlib.nullcontext()
    with ctx, torch.no_grad():
        a, b = j._pair(ref, text, prior)
        enc = j.tok([a], [b], truncation='only_first', max_length=j.max_len, return_tensors='pt').to(j.device)
        with torch.autocast(j.device, enabled=(j.device == 'cuda')):
            return float(j.model(**enc).logits.squeeze().float().cpu())

POOL = ThreadPoolExecutor(2)
def judge_par(jF, jI, refF, refI, x):
    fF = POOL.submit(_score_on_stream, jF, refF, x['text'], x['prior'])
    fI = POOL.submit(_score_on_stream, jI, refI, x['text'], x['prior'])
    return fF.result(), fI.result()

def bench(fn, jF, jI, inputs, refs, warm=10):
    for x in inputs[:warm]: fn(jF, jI, *refs[x['ref']], x)
    ms, sc = [], []
    for x in inputs:
        t0 = time.perf_counter()
        sf, si = fn(jF, jI, *refs[x['ref']], x)
        c = combine(sf, si)
        ms.append((time.perf_counter() - t0) * 1000); sc.append((sf, si, c))
    return np.array(ms), np.array(sc)

def bench_one(j, inputs, refs, which, warm=10):
    for x in inputs[:warm]: j.judge(refs[x['ref']][which], x['text'], x['prior'])
    return np.array([j.judge(refs[x['ref']][which], x['text'], x['prior']).judge_ms for x in inputs])

jF, jI = make_judge(F_DIR, DEVICE), make_judge(I_DIR, DEVICE)
assert jF.mode == 'retrieval' and jI.mode == 'full', (jF.mode, jI.mode)
refs = {}
for x in INPUTS:
    if x['ref'] not in refs: refs[x['ref']] = (jF.prepare(x['ref']), jI.prepare(x['ref']))
prep_ms = [refs[r][0].prepare_ms for r in refs]

LAT = {}
for name, j, w in [('F1 base 단독', jF, 0), ('I2 단독', jI, 1)]:
    ms = bench_one(j, INPUTS, refs, w)
    LAT[name] = {'p50': float(np.percentile(ms, 50)), 'p95': float(np.percentile(ms, 95))}
ms_seq, sc_seq = bench(judge_seq, jF, jI, INPUTS, refs)
ms_par, sc_par = bench(judge_par, jF, jI, INPUTS, refs)
LAT['차례로'] = {'p50': float(np.percentile(ms_seq, 50)), 'p95': float(np.percentile(ms_seq, 95))}
LAT['동시'] = {'p50': float(np.percentile(ms_par, 50)), 'p95': float(np.percentile(ms_par, 95))}
PAR_DIFF = float(np.abs(sc_par[:, :2] - sc_seq[:, :2]).max())
for k, v in LAT.items(): print(f"{k:10} p50 {v['p50']:6.1f} / p95 {v['p95']:6.1f} ms")
print(f'동시 방식 점수 차이 최대 {PAR_DIFF:.4f} | F1 base 근거 준비 중앙값 {np.median(prep_ms):.1f}ms(요청당 1회, 판정에서 제외)')

## 4. 사전 기준 판정과 저장

In [ ]:
par_ok = PAR_DIFF <= 0.05
cands = {'차례로': LAT['차례로']['p95']}
if par_ok: cands['동시'] = LAT['동시']['p95']
BEST = min(cands, key=cands.get); P95 = cands[BEST]
IS_A100 = 'A100' in GPU
I3_4 = bool(IS_A100 and P95 <= 68)
print(f"=== 동시 방식 점수 일치: {'충족' if par_ok else '미달 → 동시 방식 제외'} (최대 차이 {PAR_DIFF:.4f}) ===")
print(f"=== I3-4: {BEST} p95 {P95:.1f}ms @ {GPU} → {'충족' if I3_4 else ('판정 보류 (A100 아님)' if not IS_A100 else '미달')} ===")
summ = {'gpu': GPU, 'latency': LAT, 'par_max_diff': PAR_DIFF, 'best': BEST, 'p95': P95, 'I3_4': I3_4, 'is_a100': IS_A100,
        'n_inputs': len(INPUTS), 'prepare_ms_median': float(np.median(prep_ms)), 'combo': COMBO}
LOCAL = '/content/l3_local'; os.makedirs(LOCAL, exist_ok=True)
json.dump(summ, open(os.path.join(LOCAL, 'l3_summary.json'), 'w'), ensure_ascii=False, indent=1)
np.savetxt(os.path.join(LOCAL, 'l3_ms.csv'), np.c_[ms_seq, ms_par], delimiter=',', header='seq_ms,par_ms', comments='')
for f in os.listdir(LOCAL):
    try: shutil.copy(os.path.join(LOCAL, f), os.path.join(L3_DIR, f))
    except OSError as e: print('드라이브 저장 실패', f, e)
zp = shutil.make_archive('/content/l3_results', 'zip', LOCAL)
from google.colab import files
files.download(zp)